In [1]:
import numpy as np
import pandas as pd
from datetime import timedelta
import zmq
import json

In [2]:
new_data = []

In [30]:
data={'datetime': '2016-04-03T01:27:00', 'hr': 72.0}

In [31]:
data['datetime']

'2016-04-03T01:27:00'

In [32]:
dt = pd.to_datetime(data['datetime'])
hr = data['hr']

In [33]:
new_data.append({'Time': dt, 'Value': hr})


In [34]:
new_data

[{'Time': Timestamp('2016-04-02 01:27:00'), 'Value': 72.0},
 {'Time': Timestamp('2016-04-03 01:27:00'), 'Value': 72.0}]

In [35]:
temp_df = pd.DataFrame(new_data).set_index('Time').sort_index()
current_date = dt.date()

In [36]:
temp_df 

,Value
Time,
2016-04-02 01:27:00,72.0
2016-04-03 01:27:00,72.0


In [10]:
current_date

datetime.date(2016, 4, 2)

In [11]:
seen_dates = set()

In [12]:
seen_dates.add(current_date)

In [13]:
seen_dates

{datetime.date(2016, 4, 2)}

In [14]:
# Time series feature engineering
def create_features(df):
    """
    Create time series features and lag features based on time series index.
    """
    df = df.copy()

    # Basic time-based features
    df['minute'] = df.index.minute
    df['hour'] = df.index.hour
    df['day'] = df.index.day
    df['dayofweek'] = df.index.dayofweek
    df['month'] = df.index.month

    # Lag features
    df['lag_1minute'] = df['Value'].shift(1)  # 1 minute lag
    df['lag_1h'] = df['Value'].shift(60)   # 1 hour lag
    df['lag_1d'] = df['Value'].shift(1440)  # 1 day lag

    # Rolling statistics features
    df['rolling_mean_30minutes'] = df['Value'].rolling(window=30).mean()  # Last 30 minutes rolling mean
    df['rolling_mean_3hours'] = df['Value'].rolling(window=180).mean()  # Last 3 hours rolling mean
    df['rolling_mean_1days'] = df['Value'].rolling(window=1440).mean()  # Last 1 day rolling mean
    df['rolling_mean_same_hour_last_day'] = df['Value'].shift(1440).rolling(window=30).mean()  # Same hour previous day rolling mean

    return df

In [15]:
def prepare_features(df):
    """Prepare advanced features matching our app_model_xgb."""
    df = df.copy()
    df = create_features(df)  # Using our existing create_features
    return df.dropna()

In [16]:
train_features = create_features(temp_df)

In [17]:
temp_df

,Value
Time,
2016-04-02 01:27:00,72.0


In [37]:
len(temp_df)

2

In [18]:
train_features

,Value,minute,hour,day,dayofweek,month,lag_1minute,lag_1h,lag_1d,rolling_mean_30minutes,rolling_mean_3hours,rolling_mean_1days,rolling_mean_same_hour_last_day
Time,,,,,,,,,,,,,
2016-04-02 01:27:00,72.0,27,1,2,5,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [19]:
TARGET = 'Value'
FEATURES_XGB = [
    'hour', 'dayofweek', 'month', 'minute', 'day', 'lag_1minute', 'lag_1h', 'lag_1d',
    'rolling_mean_30minutes', 'rolling_mean_3hours', 'rolling_mean_1days', 'rolling_mean_same_hour_last_day'
]

In [20]:
X_train = train_features[FEATURES_XGB]
y_train = temp_df.loc[X_train.index, TARGET]

In [21]:
X_train

,hour,dayofweek,month,minute,day,lag_1minute,lag_1h,lag_1d,rolling_mean_30minutes,rolling_mean_3hours,rolling_mean_1days,rolling_mean_same_hour_last_day
Time,,,,,,,,,,,,
2016-04-02 01:27:00,1,5,4,27,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [22]:
y_train

Time
2016-04-02 01:27:00    72.0
Name: Value, dtype: float64

In [23]:
# Load the dataset
df = pd.read_csv('cardio_train.csv', sep=';')

In [24]:
# Calculate age in years
df['age_years'] = df['age'] / 365

In [25]:
df

,id,age,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio,age_years
0,0,18393,2,168,62.0,110,80,1,1,0,0,1,0,50.391781
1,1,20228,1,156,85.0,140,90,3,1,0,0,1,1,55.419178
2,2,18857,1,165,64.0,130,70,3,1,0,0,0,1,51.663014
3,3,17623,2,169,82.0,150,100,1,1,0,0,1,1,48.282192
4,4,17474,1,156,56.0,100,60,1,1,0,0,0,0,47.873973
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
69995,99993,19240,2,168,76.0,120,80,1,1,1,0,1,0,52.712329
69996,99995,22601,1,158,126.0,140,90,2,2,0,0,1,1,61.920548
69997,99996,19066,2,183,105.0,180,90,3,1,0,1,0,1,52.235616
69998,99998,22431,1,163,72.0,135,80,1,2,0,0,0,1,61.454795


In [26]:
# info 
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 70000 entries, 0 to 69999
Data columns (total 14 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   id           70000 non-null  int64  
 1   age          70000 non-null  int64  
 2   gender       70000 non-null  int64  
 3   height       70000 non-null  int64  
 4   weight       70000 non-null  float64
 5   ap_hi        70000 non-null  int64  
 6   ap_lo        70000 non-null  int64  
 7   cholesterol  70000 non-null  int64  
 8   gluc         70000 non-null  int64  
 9   smoke        70000 non-null  int64  
 10  alco         70000 non-null  int64  
 11  active       70000 non-null  int64  
 12  cardio       70000 non-null  int64  
 13  age_years    70000 non-null  float64
dtypes: float64(2), int64(12)
memory usage: 7.5 MB


In [27]:
df.describe()

,id,age,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio,age_years
count,70000.000000,70000.000000,70000.000000,70000.000000,70000.000000,70000.000000,70000.000000,70000.000000,70000.000000,70000.000000,70000.000000,70000.000000,70000.000000,70000.000000
mean,49972.419900,19468.865814,1.349571,164.359229,74.205690,128.817286,96.630414,1.366871,1.226457,0.088129,0.053771,0.803729,0.499700,53.339358
std,28851.302323,2467.251667,0.476838,8.210126,14.395757,154.011419,188.472530,0.680250,0.572270,0.283484,0.225568,0.397179,0.500003,6.759594
min,0.000000,10798.000000,1.000000,55.000000,10.000000,-150.000000,-70.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,29.583562
25%,25006.750000,17664.000000,1.000000,159.000000,65.000000,120.000000,80.000000,1.000000,1.000000,0.000000,0.000000,1.000000,0.000000,48.394521
50%,50001.500000,19703.000000,1.000000,165.000000,72.000000,120.000000,80.000000,1.000000,1.000000,0.000000,0.000000,1.000000,0.000000,53.980822
75%,74889.250000,21327.000000,2.000000,170.000000,82.000000,140.000000,90.000000,2.000000,1.000000,0.000000,0.000000,1.000000,1.000000,58.430137
max,99999.000000,23713.000000,2.000000,250.000000,200.000000,16020.000000,11000.000000,3.000000,3.000000,1.000000,1.000000,1.000000,1.000000,64.967123


In [28]:
import matplotlib.pyplot as plt
import seaborn as sns
# 2. Visualization


KeyboardInterrupt: 